[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-IA_Ed2026/blob/main/meca-ia/C19/E19-2_Diagnostic_Anomalies.ipynb)

# Diagnostic d'anomalies en service : un capteur change, le modèle dérive

## Le contexte

Six pompes d'un parc sont relevées heure par heure : débit, pression, vibration, courant du moteur et température du carter. Quand une pompe s'écarte de son fonctionnement normal pendant quelques heures, l'équipe de maintenance l'inspecte et note une anomalie. Un modèle appris sur le début du suivi doit aider l'équipe en service. Mais au bout de 540 heures, le capteur de vibration de chaque pompe a été remplacé : le nouveau lit en moyenne 0,5 mm/s de plus que l'ancien, anomalie ou non. Le modèle va-t-il continuer à tenir ?

La grandeur à prédire est l'état d'une heure, normal ou anormal : c'est un problème de classification déséquilibré sur des séries temporelles. Les anomalies sont rares : 228 relevés sur 4 800, soit 4,8 %. La base de test est la fin de chaque pompe, son dernier quart de relevés : elle joue le rôle de la mise en service, après le changement de capteur.

Défi : prédire `anomalie` et obtenir un F-score macro supérieur à celui du modèle de départ sur la base de test. Le F-score macro est la moyenne des F-scores des deux classes : il pèse autant la classe rare que la classe fréquente. Le modèle de départ est une régression logistique à poids de classes équilibrés sur les capteurs et l'identité de la pompe ; il atteint 0,581. Deux comparaisons sans apprentissage : ne jamais signaler (0,488) et une carte de contrôle de la vibration, dont la limite supérieure est la moyenne des relevés normaux de l'apprentissage plus trois écarts types (0,769). À vous de faire mieux. La base de test sert à mesurer, jamais à apprendre ni à régler.

## Les données

Fichier `MecaIA-Diagnostic_Anomalies.xlsx`, dans le dossier `donnees` du dépôt : 4 800 relevés, 8 colonnes. Une ligne est le relevé horaire d'une pompe du parc.

| Colonne | Signification | Unité |
|---|---|---|
| `horodatage` | heure pleine du relevé, 800 relevés par pompe | date et heure |
| `machine` | identifiant de la pompe, `p1` à `p6` | catégorie |
| `debit_m3_h` | débit moyen de l'heure | m³/h |
| `pression_bar` | pression de refoulement (quelques cases vides) | bar |
| `vibration_mm_s` | vitesse de vibration efficace ; capteur changé à l'heure 540, avec un décalage | mm/s |
| `courant_A` | courant du moteur | A |
| `temperature_C` | température du carter (quelques cases vides) | °C |
| `anomalie` | cible : 1 si l'équipe de maintenance a diagnostiqué une anomalie pendant l'heure | 0 ou 1 |

Origine : jeu simulé, six pompes à épisodes d'anomalie de quelques heures et changement du capteur de vibration à l'heure 540. Il enseigne une méthode, et aucune conclusion de maintenance n'en découle.

### Étape 1 : charger le fichier et regarder sa forme

On lit le classeur, puis on vérifie ses dimensions, ses cases vides, le pas de temps, le nombre de relevés par pompe et la part d'anomalies : la forme d'un fichier se lit sur le fichier entier. Le dossier des données est cherché depuis le dossier du notebook, puis depuis un clone du dépôt ; ouvert seul, par exemple dans Colab, le notebook télécharge une fois le jeu depuis le dépôt public.

In [ ]:
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import set_config

set_config(display="text")
GRAINE = 42
FICHIER = "MecaIA-Diagnostic_Anomalies.xlsx"
URL_JEU = "https://raw.githubusercontent.com/philippe-amuzuga/Meca-IA_Ed2026/main/donnees/MecaIA-Diagnostic_Anomalies.xlsx"

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-IA_Ed2026", "donnees")]
dossier = next((d for d in candidats if (d / FICHIER).is_file()), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique du jeu depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(URL_JEU, dossier / FICHIER)
    except OSError as erreur:
        raise FileNotFoundError(
            "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
df = pd.read_excel(dossier / FICHIER)

cible = "anomalie"
vides = df.isna().sum()
print("dimensions :", df.shape)
print("cases vides :", vides[vides > 0].to_dict())
pas = df.groupby("machine")["horodatage"].diff().dropna()
print("pas de temps :", {str(k): int(n) for k, n in pas.value_counts().items()})
print("relevés par pompe :", sorted(df["machine"].value_counts().unique().tolist()))
print("anomalies :", int(df[cible].sum()), "relevés sur", len(df), "; par pompe de", int(df.groupby("machine")[cible].sum().min()), "à", int(df.groupby("machine")[cible].sum().max()))
df.head(3)

Lecture du résultat : 4 800 relevés de 6 pompes, au pas d'une heure. Deux colonnes ont des cases vides : la pression (95) et la température (89). Les anomalies sont rares, 4,8 % des relevés : un modèle qui ne signalerait jamais rien aurait une exactitude de 95,2 %, et ne dirait rien. Chaque pompe connaît de 31 à 49 heures d'anomalie.

### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas le dossier `donnees/` du dépôt. La cellule suivante ne télécharge rien : elle cherche ce dossier (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-IA_Ed2026`), vérifie la présence du classeur du défi et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez le dossier `donnees` à la racine de l'espace de travail et téléversez-y le classeur listé, pris dans le dépôt.

In [ ]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
from pathlib import Path

JEUX_LUS = ["MecaIA-Diagnostic_Anomalies.xlsx"]                  # le classeur lu par ce notebook


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-IA_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees = chercher("donnees")
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Étape 2 : ranger le temps et mettre le test de côté

On trie les relevés de chaque pompe par date. Le dernier quart de chaque pompe est mis de côté avant tout apprentissage : il joue le rôle de l'avenir en service. Aucun mélange des lignes n'est permis.

In [ ]:
df = df.sort_values(["machine", "horodatage"])
par_machine = df.groupby("machine")
effectif = par_machine["horodatage"].transform("size")
rang_depuis_la_fin = par_machine.cumcount(ascending=False)           # 0 : dernier relevé de la pompe
est_test = rang_depuis_la_fin < np.ceil(0.25 * effectif)
capteurs = ["debit_m3_h", "pression_bar", "vibration_mm_s", "courant_A", "temperature_C"]
X = df[["machine"] + capteurs]
y = df[cible]
X_train, X_test, y_train, y_test = X[~est_test], X[est_test], y[~est_test], y[est_test]
print("apprentissage :", len(X_train), "relevés dont", int(y_train.sum()), "anomalies")
print("test :", len(X_test), "relevés dont", int(y_test.sum()), "anomalies")

### Étape 3 : lire la dérive sur l'apprentissage, avec une carte de contrôle

Une carte de contrôle fixe la plage normale d'une grandeur, ici la vibration : la moyenne des relevés normaux plus ou moins trois écarts types. On la trace sur l'apprentissage, par tranches de 50 heures, avec la médiane de la vibration des seuls relevés normaux. Si le procédé est stable, la médiane reste dans la plage ; si elle en sort, le capteur ou le procédé a changé.

In [ ]:
normaux = X_train[y_train == 0]
centre, sigma = normaux["vibration_mm_s"].mean(), normaux["vibration_mm_s"].std()
heure = X_train.groupby("machine").cumcount()
mediane_par_tranche = normaux["vibration_mm_s"].groupby((heure[normaux.index] // 50) * 50).median()
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.plot(mediane_par_tranche.index, mediane_par_tranche, "o-", color="0.15", label="médiane des relevés normaux")
ax.axhline(centre, color="0.4", linestyle="--", label="moyenne des relevés normaux")
ax.axhline(centre + 3 * sigma, color="0.4", linestyle=":", label="limite à trois écarts types")
ax.set(xlabel="heure de service (début de tranche de 50 h)", ylabel="vibration (mm/s)")
ax.legend(frameon=False, fontsize=7)
ax.grid(alpha=0.3)
plt.show()
print(mediane_par_tranche.round(2).to_string())

Lecture du résultat : la médiane de la vibration des relevés normaux reste voisine de 2,91 mm/s jusqu'à l'heure 500, puis monte à 3,33 mm/s sur la dernière tranche, soit 0,43 mm/s de plus. La limite de la carte est à 4,52 mm/s, plus haute que la vibration d'une pompe normale, car elle regroupe toutes les pompes, dont la vibration saine propre va de 1,8 à 3,2 mm/s. Le saut est net, mais la carte globale ne le voit pas : il faut comparer chaque pompe à elle-même.

### Étape 4 : construire le modèle de départ

Le modèle de départ est une régression logistique sur les cinq capteurs et l'identité de la pompe. Les cases vides de la pression et de la température sont remplacées par la médiane de l'apprentissage, chaque capteur est ramené à une moyenne nulle et à un écart type de 1, et `class_weight="balanced"` donne à la classe rare un poids proportionnel à sa rareté. Le pipeline apprend sur `X_train` seul.

In [ ]:
# Votre code :

### Étape 5 : mesurer sur la base de test

Le modèle diagnostique des heures qu'il n'a jamais vues, après le changement de capteur. On lit le F-score macro, la précision (part des alertes justifiées) et le rappel (part des anomalies signalées) de la classe rare, et la matrice de confusion. Deux comparaisons sans apprentissage : ne jamais signaler, et la carte de contrôle de la vibration.

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

y_pred = modele.predict(X_test)
score_depart = f1_score(y_test, y_pred, average="macro")
score_jamais = f1_score(y_test, np.zeros(len(y_test), dtype=int), average="macro")
limite = centre + 3 * sigma
score_carte = f1_score(y_test, (X_test["vibration_mm_s"] > limite).astype(int), average="macro")
print(f"F-score macro : {score_depart:.3f}")
print(f"précision de la classe rare : {precision_score(y_test, y_pred):.3f}, rappel : {recall_score(y_test, y_pred):.3f}")
print(f"F-score macro, ne jamais signaler : {score_jamais:.3f}")
print(f"F-score macro, carte de contrôle (limite {limite:.2f} mm/s) : {score_carte:.3f}")
print("matrice de confusion (lignes : réel, colonnes : prédit ; normal, anomalie)")
print(confusion_matrix(y_test, y_pred))

Lecture du résultat : le F-score macro est de 0,581, contre 0,488 pour le témoin qui ne signale jamais rien et 0,769 pour la carte de contrôle. Le rappel de la classe rare est de 1,000 mais la précision de 0,175 : le modèle signale 273 heures normales sur 1142 comme des anomalies. Ce sont de fausses alertes dues au changement de capteur : le modèle a appris que la vibration haute annonce l'anomalie, et la vibration est devenue plus haute pour toutes les heures. La carte de contrôle, qui n'a aucun apprentissage, fait mieux que lui.

## Pour aller plus loin : comparer chaque pompe à elle-même, et décider du réentraînement

Le modèle de départ de l'étape 4 reste le score de référence du défi. Les cellules suivantes montrent comment le dépasser sans regarder le test pour choisir. La première ajoute, pour la vibration, le courant et la température, l'écart de chaque relevé à la médiane des 48 heures précédentes de la même pompe : un décalage de capteur, qui déplace la médiane, n'apparaît plus dans l'écart. Elle compare ensuite, sur une validation chronologique (les 25 % les plus récents de l'apprentissage de chaque pompe, qui contiennent le changement de capteur), la régression logistique et un gradient boosting, avec et sans ces écarts. La deuxième suit le taux d'alarmes du modèle de départ par tranches de 50 heures de validation et le compare aux limites d'une carte de contrôle de proportions : c'est le critère qui décide d'un réentraînement. La dernière mesure le modèle retenu une seule fois sur le test.

In [ ]:
# Votre code :

In [ ]:
# Votre code :

In [ ]:
# Votre code :

Lecture du résultat : sur la validation chronologique, la régression logistique de départ atteint 0,664 et le gradient boosting sur les capteurs 0,709 ; avec les écarts à la pompe, la régression logistique monte à 0,731 et le gradient boosting à 0,821 : le modèle retenu est « gradient boosting avec écarts à la pompe ». Mesuré une seule fois sur le test, il donne un F-score macro de 0,955, soit 0,374 de plus que le modèle de départ. La carte de proportions dit quand réentraîner : le taux d'alarmes du modèle de départ passe de 0,050 sur la période d'ajustement à 0,357 sur la dernière tranche, et 2 tranches de validation sur 3 sortent de la limite haute : le critère est atteint dès le changement de capteur, avant que l'équipe n'ait étiqueté une seule heure. Réentraîner ne suffit pas toujours : ici, c'est le choix des variables, relatives à chaque pompe, qui rend le modèle indifférent au décalage.

## Le score de référence

Le score publié est le F-score macro du modèle de départ de l'étape 4 sur la base de test. L'empreinte identifie cette base de test : si la vôtre diffère, le découpage n'est pas le même et les scores ne se comparent pas.

In [ ]:
import hashlib

score = f1_score(y_test, modele.predict(X_test), average="macro")
positions = ",".join(str(i) for i in sorted(X_test.index))
print("EMPREINTE_TEST =", hashlib.sha256(positions.encode("ascii")).hexdigest())
print("SCORE_REFERENCE f1_macro = %.3f" % score)

## Synthèse

1. Un changement de capteur déplace les entrées sans qu'aucune pompe ne se dégrade : la carte de contrôle de la vibration le montre sur l'apprentissage, par la médiane des relevés normaux.
2. Un modèle appris sur les valeurs absolues s'effondre après le changement : 0,581 de F-score macro sur la base de test, moins que la carte de contrôle (0,769), parce qu'il signale presque toutes les heures normales.
3. Comparer chaque relevé à la médiane récente de la même pompe rend le modèle indifférent au décalage : sur la validation chronologique, le gradient boosting passe de 0,709 à 0,821.
4. Le taux d'alarmes, suivi sur une carte de proportions, dit quand réentraîner sans attendre les étiquettes : il sort de sa limite haute dès le changement de capteur.
5. Mesuré une seule fois sur le test, le modèle retenu donne 0,955.